# Predictive Manufacturing — Starter Notebook

This notebook is a zero-opinion starting point. It only loads the hackathon dataset and shows you what's inside each sheet. From here on, the analysis is yours.

**Before you run:**
1. Make sure you've `pip install -r requirements.txt`
2. Confirm `data/hackathon_dataset.xlsx` exists

**What this notebook does:**
- Loads every sheet (headers are already clean on row 1 — no gymnastics needed)
- Prints each sheet's shape and column list
- Shows a small sample from each
- Runs basic join sanity checks so you can confirm the data resolves correctly on your machine

**What this notebook does NOT do:**
- Any analysis, modeling, scenario simulation, or reporting. That's your job.

## 1. Imports

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

DATA_PATH = Path('..') / 'data' / 'hackathon_dataset.xlsx'
assert DATA_PATH.exists(), f'Dataset not found at {DATA_PATH.resolve()}'
print('Loading:', DATA_PATH.resolve())

## 2. Load every sheet

The dataset is already cleaned: each sheet has a single-row header on row 1, no blank leading columns, no preamble metadata. `pd.read_excel` with default settings is enough.

In [ ]:
SHEETS = [
    'Flow',
    '1_1 Export Plates',
    '1_2 Gaskets',
    '1_3 Export Project list',
    '2_1 Work Center Capacity Weekly',
    '2_2 OPS plan per material',
    '2_3 SAP MasterData',
    '2_4 Model Calendar',
    '2_5 WC Schedule_limits',
    '2_6 Tool_material nr master',
    '3_1 Inventory ATP',
    '3_2 Component_SF_RM',
]

sheets = {name: pd.read_excel(DATA_PATH, sheet_name=name) for name in SHEETS}

for name, df in sheets.items():
    print(f'{name!r:40s} shape={df.shape}')

## 3. Peek at each sheet

In [ ]:
for name, df in sheets.items():
    print(f'\n=== {name} ===')
    print('columns:', list(df.columns)[:12], '...' if len(df.columns) > 12 else '')
    print(df.head(3).to_string(max_cols=10))

## 4. Quick join sanity checks

If these numbers look wrong, something about your ingest is off.

In [ ]:
df_plates = sheets['1_1 Export Plates']
df_gaskets = sheets['1_2 Gaskets']
df_26 = sheets['2_6 Tool_material nr master']
df_32 = sheets['3_2 Component_SF_RM']
df_31 = sheets['3_1 Inventory ATP']
df_13 = sheets['1_3 Export Project list']
df_21 = sheets['2_1 Work Center Capacity Weekly']

pipeline_connectors = set(
    df_plates['Connector Plant_Material nr'].dropna().astype(str)
) | set(df_gaskets['Connector Plant_Material nr'].dropna().astype(str))
pipeline_connectors.discard('_')
master_connectors = set(df_26['Connector'].dropna().astype(str))

print('Unique pipeline connectors (excluding `_`):', len(pipeline_connectors))
print('  matched in 2_6:', len(pipeline_connectors & master_connectors))

pipeline_projects = set(df_plates['Project_name'].dropna().astype(str)) | set(
    df_gaskets['Project_name'].dropna().astype(str)
)
project_list = set(df_13['Project name'].dropna().astype(str))
print('Pipeline projects matched to 1_3 project list:',
      len(pipeline_projects & project_list), '/', len(pipeline_projects))

bom_headers = set(df_32['Header Material code'].dropna().astype(str))
master_saps = set(df_26['Sap code'].dropna().astype(str))
print('3_2 BOM headers found in 2_6 Sap codes:',
      len(bom_headers & master_saps), '/', len(bom_headers))

inventory_codes = set(df_31['Material Unique (code)'].dropna().astype(str))
print('Inventory material codes overlapping with 2_6:',
      len(inventory_codes & master_saps), '/', len(inventory_codes))

wc_21 = set(df_21['Work center code'].dropna().astype(str))
wc_from_26 = {f"P01_{p}_{w}" for p, w in zip(df_26['Plant'].astype(str), df_26['Work center'].astype(str)) if str(w) not in ('nan', '#N/A')}
print('2_6 work-center codes matched in 2_1:', len(wc_from_26 & wc_21), '/', len(wc_from_26))

## 5. Data-quality surface scan

Every one of these is a decision point for your team. Drop? Impute? Flag? Your call.

In [ ]:
print('Rows with `_` connector in 1_1:',
      int((df_plates['Connector Plant_Material nr'] == '_').sum()))
print('Rows with `_` connector in 1_2:',
      int((df_gaskets['Connector Plant_Material nr'] == '_').sum()))
print('Rows with "Missing CT" in 1_1:',
      int((df_plates['Cycle time'] == 'Missing CT').sum()))

# #N/A in 2_6.Work center is read as NaN by default
wc_na = df_26['Work center'].isna().sum()
print('2_6 rows with NaN (originally #N/A) Work center:', int(wc_na))

rev_by_mat = df_26.groupby('Sap code')['Rev no'].nunique()
print('Materials with >1 Rev no across plants:',
      int((rev_by_mat > 1).sum()))

cross_plant = df_26.groupby('Tool No.')['Plant'].nunique()
print('Tools present at 2+ plants (substitution candidates):',
      int((cross_plant > 1).sum()))

## 6. Where to go from here

A few prompts for your team — pick the ones that interest you:

- Build a single joined dataframe at `(plant, work_center, tool, material, month)` grain for a pilot factory.
- Pick a work center and compare `Final Operations Plan, Load in hours` (from 2_1) against `Available Capacity, hours`. Where is the headroom?
- Layer pipeline demand from `1_1` / `1_2` on top. For which work centers does it push total load above the available capacity line?
- For the same bottleneck, check if the same tool number exists at a different plant (via 2_6) — is there a re-allocation option?
- For a single pipeline project, walk the BOM in `3_2` down to raw material, then compare against `3_1` inventory. When do you need to place an order to meet the demand date?
- Think about how you want to handle probability. The data gives you a per-project probability in `1_3`. What does 'expected capacity utilization' mean in a world where only some of these projects close?

Good luck. Build something you're proud to demo.